# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aneeqahabib/FlyRank_ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Signal checks and my rule

This lane is **Ranking Signal Analysis**, so the baseline is a transparent content-review priority queue. I check two signals before writing the rule:

1. **Staleness:** `days_since_last_update`, the signal behind FlyRank's refresh flags. Pages at least 180 days from their last update are candidates for review, but the bucket table checks whether that pattern is actually visible in this slice.
2. **Prior-window visibility:** `impressions_prev_30d`, a safe demand/visibility measure from days 31–60 before the observed last-30-day outcome. The rule uses this earlier window rather than 90-day totals or last-30-day fields.

**Verdicts:** the code prints `CONFIRMED`, `MIXED`, `OPPOSITE`, or `FALSE` from the observed bucket comparisons. These are associations for decision support, not causal claims.

**One rule:** add fixed points for staleness and prior-window visibility. A page with at least 180 days since update and at least 100 prior-window impressions gets the reason code `stale_visible_refresh` and action `refresh`. Other high-volume pages get `high_visibility_review`; older but low-visibility pages get `stale_low_visibility`; the remainder gets `monitor`. Ties are broken deterministically by `content_id`.

In [6]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
QUEUE_PATH = Path("../outputs/baseline_action_score.csv")
RECEIPT_PATH = Path("../outputs/baseline_action_score_metrics.json")

df = pd.read_csv(DATA_PATH)
required_columns = {
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_prev_30d",
    "trend_direction",
}
missing_columns = required_columns - set(df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

# Keep the observed label only for signal auditing and evaluation, never for scoring.
df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)
df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
).fillna(0)
df["impressions_prev_30d"] = pd.to_numeric(
    df["impressions_prev_30d"], errors="coerce"
).fillna(0)

staleness_bins = [-np.inf, 90, 180, 365, np.inf]
staleness_labels = ["0-90", "91-180", "181-365", "365+"]
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=staleness_bins,
    labels=staleness_labels,
    right=True,
).astype(str)

volume_bins = [-np.inf, 0, 99, 499, 1999, np.inf]
volume_labels = ["0", "1-99", "100-499", "500-1999", "2000+"]
df["prior_visibility_bucket"] = pd.cut(
    df["impressions_prev_30d"],
    bins=volume_bins,
    labels=volume_labels,
    right=True,
).astype(str)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_prior_impressions=("impressions_prev_30d", "median"),
        observed_decline_rate=("is_declining_label", "mean"),
    )
    .reindex(staleness_labels)
    .reset_index()
)
volume_table = (
    df.groupby("prior_visibility_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_days_since_update=("days_since_last_update", "median"),
        observed_decline_rate=("is_declining_label", "mean"),
    )
    .reindex(volume_labels)
    .reset_index()
)

print("Staleness signal bucket table (flag-linked signal):")
display(staleness_table)
print("Prior-window visibility bucket table:")
display(volume_table)

stale_high_rate = staleness_table.loc[
    staleness_table["staleness_bucket"].isin(["181-365", "365+"]),
    "observed_decline_rate",
].mean()
stale_low_rate = staleness_table.loc[
    staleness_table["staleness_bucket"].isin(["0-90", "91-180"]),
    "observed_decline_rate",
].mean()
volume_high_rate = volume_table.loc[
    volume_table["prior_visibility_bucket"].isin(["500-1999", "2000+"]),
    "observed_decline_rate",
].mean()
volume_low_rate = volume_table.loc[
    volume_table["prior_visibility_bucket"].isin(["0", "1-99"]),
    "observed_decline_rate",
].mean()

staleness_verdict = "CONFIRMED" if stale_high_rate > stale_low_rate else "OPPOSITE" if stale_high_rate < stale_low_rate else "MIXED"
volume_verdict = "CONFIRMED" if volume_high_rate > volume_low_rate else "OPPOSITE" if volume_high_rate < volume_low_rate else "MIXED"
print(f"Staleness verdict: {staleness_verdict} (older={stale_high_rate:.3f}, newer={stale_low_rate:.3f})")
print(f"Prior-window visibility verdict: {volume_verdict} (higher={volume_high_rate:.3f}, lower={volume_low_rate:.3f})")

# Hand-written, fixed weights: no fitted model and no label-derived input.
stale_points = np.select(
    [df["days_since_last_update"] >= 365, df["days_since_last_update"] >= 180],
    [2, 1],
    default=0,
)
visibility_points = np.select(
    [df["impressions_prev_30d"] >= 2000, df["impressions_prev_30d"] >= 100],
    [2, 1],
    default=0,
)
df["baseline_score"] = stale_points + visibility_points
df["reason_code"] = np.select(
    [
        (stale_points >= 1) & (visibility_points >= 1),
        (visibility_points >= 2),
        (stale_points >= 1),
    ],
    ["stale_visible_refresh", "high_visibility_review", "stale_low_visibility"],
    default="monitor",
)
df["action"] = np.select(
    [df["reason_code"].eq("stale_visible_refresh"), df["baseline_score"] >= 2],
    ["refresh", "review"],
    default="monitor",
)

sorted_indices = df.sort_values(
    ["baseline_score", "content_id"], ascending=[False, True]
).index
df.loc[sorted_indices, "baseline_rank"] = np.arange(1, len(df) + 1)
df["baseline_rank"] = df["baseline_rank"].astype(int)

assert not df["reason_code"].str.contains(r"\|").any()
assert set(df["action"]).issubset({"refresh", "review", "monitor"})
print("Rule encoded. Candidate rows:", int((df["baseline_score"] > 0).sum()))
print("Action counts:")
print(df["action"].value_counts().to_string())

Staleness signal bucket table (flag-linked signal):


,staleness_bucket,n,median_prior_impressions,observed_decline_rate
0,0-90,20655,128.0,0.512031
1,91-180,9171,492.0,0.611057
2,181-365,169,5.0,0.467456
3,365+,5,2.0,0.600000


Prior-window visibility bucket table:


,prior_visibility_bucket,n,median_days_since_update,observed_decline_rate
0,0,3388,20.0,0.000000
1,1-99,8602,20.0,0.601256
2,100-499,6891,22.0,0.628066
3,500-1999,5855,22.0,0.642186
4,2000+,5264,25.0,0.570289


Staleness verdict: OPPOSITE (older=0.534, newer=0.562)
Prior-window visibility verdict: CONFIRMED (higher=0.606, lower=0.301)
Rule encoded. Candidate rows: 18163
Action counts:
action
monitor    24717
review      5262
refresh       21


## 2. Build the ranked queue (writes the CSV)

The queue is deterministic and auditable: every row has one numeric score, one reason code, one action label, and a stable rank. The observed decline label is used only to report the baseline's retrospective precision, never to calculate the queue.

In [7]:
queue_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_prev_30d",
]
queue = df.sort_values("baseline_rank")[queue_columns].copy()
QUEUE_PATH.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(QUEUE_PATH, index=False)

# Precision@K is a retrospective diagnostic against the observed label, not a training input.
precision_at_k = {}
for k in [10, 50, 100]:
    top_k = df.sort_values("baseline_rank").head(k)
    precision_at_k[f"precision_at_{k}"] = float(top_k["is_declining_label"].mean())

metrics_receipt = {
    "rows_ranked": int(len(queue)),
    "base_rate_observed_decline": float(df["is_declining_label"].mean()),
    "precision_at_k": precision_at_k,
    "staleness_verdict": staleness_verdict,
    "prior_visibility_verdict": volume_verdict,
    "score_formula": "stale_points + visibility_points; fixed hand-written thresholds",
    "scoring_inputs": ["days_since_last_update", "impressions_prev_30d"],
    "excluded_from_scoring": [
        "trend_direction",
        "trend_pct",
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "content_id",
        "client_id",
    ],
}
RECEIPT_PATH.write_text(json.dumps(metrics_receipt, indent=2))

print(f"Wrote ranked queue: {QUEUE_PATH}")
print(f"Wrote metrics receipt: {RECEIPT_PATH}")
print("Observed decline base rate:", round(metrics_receipt["base_rate_observed_decline"], 3))
print("Precision@K:", {key: round(value, 3) for key, value in precision_at_k.items()})
display(queue.head(10))

Wrote ranked queue: ..\outputs\baseline_action_score.csv
Wrote metrics receipt: ..\outputs\baseline_action_score_metrics.json
Observed decline base rate: 0.542
Precision@K: {'precision_at_10': 0.8, 'precision_at_50': 0.62, 'precision_at_100': 0.57}


,content_id,client_id,baseline_rank,baseline_score,reason_code,action,days_since_last_update,impressions_prev_30d
21268,content_0a91db491d14,client_7f2253d7e2,1,3,stale_visible_refresh,refresh,193,4662
7021,content_1bfaa38ff26c,client_7f2253d7e2,2,3,stale_visible_refresh,refresh,194,9101
11489,content_5feee3994adb,client_7f2253d7e2,3,3,stale_visible_refresh,refresh,194,2670
16514,content_7368877ea310,client_7f2253d7e2,4,3,stale_visible_refresh,refresh,194,20472
12045,content_c2d929d83eaa,client_7f2253d7e2,5,3,stale_visible_refresh,refresh,193,2160
16751,content_cf56e2e2e282,client_7f2253d7e2,6,3,stale_visible_refresh,refresh,194,26791
26810,content_ecb6215e79fd,client_7f2253d7e2,7,3,stale_visible_refresh,refresh,194,2142
7022,content_00144b1e6482,client_6208ef0f77,8,2,high_visibility_review,review,104,4647
3208,content_001ae9124afc,client_6208ef0f77,9,2,high_visibility_review,review,104,13989
16071,content_001be51d94db,client_19581e27de,10,2,high_visibility_review,review,22,3649


## 3. Top-10 review

The top ten are not accepted as truth. Each row gets an action, the rule's reason, and a concrete condition that would make the recommendation wrong. The repeated client concentration in this small top slice is also a caution: the queue ranks pages, but it should be checked for client-level concentration before operational use.

In [8]:
top10 = queue.head(10).copy()

def review_why(row: pd.Series) -> str:
    return (
        f"score {int(row['baseline_score'])}: {row['days_since_last_update']:.0f} days since update "
        f"and {row['impressions_prev_30d']:.0f} prior-window impressions"
    )


def review_risk(row: pd.Series) -> str:
    if row["reason_code"] == "stale_visible_refresh":
        return "The page may still be relevant and stable; staleness plus prior visibility does not prove a refresh will help."
    if row["reason_code"] == "high_visibility_review":
        return "High prior visibility may reflect a topic that is already healthy; the queue has no quality or intent-change evidence."
    if row["reason_code"] == "stale_low_visibility":
        return "Low prior visibility may mean the page has little opportunity, so refreshing it could waste effort."
    return "The monitor decision would be wrong if a hidden business priority or recent external change is not represented here."

review = top10.assign(
    why=top10.apply(review_why, axis=1),
    what_would_make_it_wrong=top10.apply(review_risk, axis=1),
)[
    [
        "baseline_rank",
        "content_id",
        "action",
        "reason_code",
        "why",
        "what_would_make_it_wrong",
    ]
]
print("Top-10 skeptical review:")
display(review)
assert len(review) == 10
assert review["what_would_make_it_wrong"].notna().all()
print("Reviewed rows:", len(review))

Top-10 skeptical review:


,baseline_rank,content_id,action,reason_code,why,what_would_make_it_wrong
21268,1,content_0a91db491d14,refresh,stale_visible_refresh,score 3: 193 days since update and 4662 prior-...,The page may still be relevant and stable; sta...
7021,2,content_1bfaa38ff26c,refresh,stale_visible_refresh,score 3: 194 days since update and 9101 prior-...,The page may still be relevant and stable; sta...
11489,3,content_5feee3994adb,refresh,stale_visible_refresh,score 3: 194 days since update and 2670 prior-...,The page may still be relevant and stable; sta...
16514,4,content_7368877ea310,refresh,stale_visible_refresh,score 3: 194 days since update and 20472 prior...,The page may still be relevant and stable; sta...
12045,5,content_c2d929d83eaa,refresh,stale_visible_refresh,score 3: 193 days since update and 2160 prior-...,The page may still be relevant and stable; sta...
16751,6,content_cf56e2e2e282,refresh,stale_visible_refresh,score 3: 194 days since update and 26791 prior...,The page may still be relevant and stable; sta...
26810,7,content_ecb6215e79fd,refresh,stale_visible_refresh,score 3: 194 days since update and 2142 prior-...,The page may still be relevant and stable; sta...
7022,8,content_00144b1e6482,review,high_visibility_review,score 2: 104 days since update and 4647 prior-...,High prior visibility may reflect a topic that...
3208,9,content_001ae9124afc,review,high_visibility_review,score 2: 104 days since update and 13989 prior...,High prior visibility may reflect a topic that...
16071,10,content_001be51d94db,review,high_visibility_review,score 2: 22 days since update and 3649 prior-w...,High prior visibility may reflect a topic that...


Reviewed rows: 10


## 4. Weak picks + leakage check

The weakest-looking positive picks are the pages with high prior visibility but no staleness evidence: they are review candidates, not automatic refreshes. More broadly, this rule is intentionally narrow. It uses only `days_since_last_update` and `impressions_prev_30d`; it does not use the observed decline label, `trend_direction`, `trend_pct`, last-30-day outcome fields, overlapping 90-day totals, CTR, position, or product flags. The queue is therefore a prioritization baseline, not a claim that these pages will decline or that refreshes cause recovery.

In [9]:
scoring_inputs = {"days_since_last_update", "impressions_prev_30d"}
forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "provider_used",
    "model_used",
}
assert scoring_inputs.isdisjoint(forbidden_inputs)
assert set(queue.columns) == {
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_prev_30d",
}
assert queue["baseline_rank"].is_unique
assert queue["baseline_rank"].tolist() == list(range(1, len(queue) + 1))
assert not queue["reason_code"].str.contains(r"\|").any()
print("Scoring inputs:", sorted(scoring_inputs))
print("Forbidden inputs absent from scoring formula:", scoring_inputs.isdisjoint(forbidden_inputs))
print("Queue ranks are unique and contiguous:", True)
print("Weak-pick caution: high visibility is a review signal, not proof that refreshing will improve performance.")

Scoring inputs: ['days_since_last_update', 'impressions_prev_30d']
Forbidden inputs absent from scoring formula: True
Queue ranks are unique and contiguous: True
Weak-pick caution: high visibility is a review signal, not proof that refreshing will improve performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.